# Reducción de dimensionalidad con PCA

## Caso de estudio: ventas fecha de semanales de Corporación Favorita

**Herramientas:** Polars + scikit-learn + Plotly  
**Archivo:** `favorita_weekly.csv`

### Objetivo

Transformar los datos de ventas fecha de semanales para que:

- cada fila represente una **tienda durante una fecha de semana identificada por su fecha de inicio**;
- cada columna represente un **producto**;
- PCA reduzca la cantidad de variables conservando la mayor parte de la información.

### Flujo

**Carga → exploración → limpieza → pivot → escalamiento → PCA → visualización**

## 1. Instalación de Polars

Google Colab ya incluye la mayoría de las librerías necesarias.  
Instalamos Polars para trabajar con el archivo de ventas.

In [1]:
!pip -q install polars plotly

## 2. Importación de librerías

In [2]:
import numpy as np
import pandas as pd
import polars as pl
import plotly.express as px

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

## 3. Subir el archivo a Google Colab

Ejecuta la siguiente celda y selecciona el archivo:

`favorita_weekly.parquet`

## 4. Leer el archivo con Polars

La lectura se realiza en una sola línea.

In [3]:
df = pl.read_parquet("/content/favorita_weekly.parquet")
df

fecha_semana,anio,semana,store_nbr,item_nbr,unit_sales,dias_promocion,dias_con_venta
datetime[ns],i16,i8,i16,i32,f32,i8,i8
2012-12-31 00:00:00,2013,1,1,103520,5.0,0,2
2012-12-31 00:00:00,2013,1,1,103665,13.0,0,5
2012-12-31 00:00:00,2013,1,1,105574,26.0,0,5
2012-12-31 00:00:00,2013,1,1,105575,38.0,0,5
2012-12-31 00:00:00,2013,1,1,105577,11.0,0,5
…,…,…,…,…,…,…,…
2017-08-14 00:00:00,2017,33,54,2109909,5.0,0,1
2017-08-14 00:00:00,2017,33,54,2110456,312.0,0,2
2017-08-14 00:00:00,2017,33,54,2113343,1.0,0,1


## 5. Revisar las primeras filas

In [4]:
df.head()

fecha_semana,anio,semana,store_nbr,item_nbr,unit_sales,dias_promocion,dias_con_venta
datetime[ns],i16,i8,i16,i32,f32,i8,i8
2012-12-31 00:00:00,2013,1,1,103520,5.0,0,2
2012-12-31 00:00:00,2013,1,1,103665,13.0,0,5
2012-12-31 00:00:00,2013,1,1,105574,26.0,0,5
2012-12-31 00:00:00,2013,1,1,105575,38.0,0,5
2012-12-31 00:00:00,2013,1,1,105577,11.0,0,5


## 6. Revisar la estructura del dataset

Observamos:

- cantidad de filas;
- cantidad de columnas;
- nombres de las variables;
- tipos de datos.

In [5]:
print("Dimensiones:", df.shape)
print("\nColumnas:")
print(df.columns)
print("\nTipos de datos:")
print(df.schema)

Dimensiones: (23582329, 8)

Columnas:
['fecha_semana', 'anio', 'semana', 'store_nbr', 'item_nbr', 'unit_sales', 'dias_promocion', 'dias_con_venta']

Tipos de datos:
Schema({'fecha_semana': Datetime(time_unit='ns', time_zone=None), 'anio': Int16, 'semana': Int8, 'store_nbr': Int16, 'item_nbr': Int32, 'unit_sales': Float32, 'dias_promocion': Int8, 'dias_con_venta': Int8})


## 7. Definir las columnas del archivo

El cuaderno utilizará los siguientes nombres:

- `fecha_fecha de semana`: fecha correspondiente al inicio de cada fecha de semana;
- `store_nbr`: tienda;
- `item_nbr`: producto;
- `unit_sales`: ventas fecha de semanales.

Si el archivo utiliza nombres distintos, modifica solamente esta celda.

In [6]:
FECHA_SEMANA = "fecha_semana"
TIENDA = "store_nbr"
PRODUCTO = "item_nbr"
VENTAS = "unit_sales"

## 8. Seleccionar las variables necesarias

Conservamos únicamente las cuatro columnas que se necesitan para construir la matriz de características.

In [7]:
df = df.select([
    pl.col(FECHA_SEMANA)
      .alias("fecha_semana"),
    pl.col(TIENDA).alias("tienda"),
    pl.col(PRODUCTO).alias("producto"),
    pl.col(VENTAS).alias("ventas")
])

df.head()

fecha_semana,tienda,producto,ventas
datetime[ns],i16,i32,f32
2012-12-31 00:00:00,1,103520,5.0
2012-12-31 00:00:00,1,103665,13.0
2012-12-31 00:00:00,1,105574,26.0
2012-12-31 00:00:00,1,105575,38.0
2012-12-31 00:00:00,1,105577,11.0


## 9. Resumen general del dataset

In [8]:
df.select([
    pl.len().alias("registros"),
    pl.col("fecha_semana").n_unique().alias("semanas"),
    pl.col("tienda").n_unique().alias("tiendas"),
    pl.col("producto").n_unique().alias("productos")
])

registros,semanas,tiendas,productos
u32,u32,u32,u32
23582329,242,54,4036


## 10. Revisar valores nulos

In [9]:
df.null_count()

fecha_semana,tienda,producto,ventas
u32,u32,u32,u32
0,0,0,0


## 11. Revisar duplicados

La llave lógica del archivo semanal es:

**fecha_fecha_fecha de semana + tienda + producto**

Como el archivo ya está consolidado semanalmente, esta combinación debería ser única.

In [10]:
duplicados = (
    df.group_by(["fecha_semana", "tienda", "producto"])
      .len()
      .filter(pl.col("len") > 1)
)

print("Combinaciones duplicadas:", duplicados.height)

Combinaciones duplicadas: 0


## 12. Limpiar y consolidar los datos

- eliminamos filas sin identificadores;
- reemplazamos ventas nulas por cero;
- agrupamos por fecha de semana, tienda y producto;
- sumamos las ventas repetidas.

In [11]:
df_limpio = (
    df.drop_nulls(["fecha_semana", "tienda", "producto"])
      .with_columns(
          pl.col("ventas").fill_null(0).cast(pl.Float32)
      )
      .group_by(["fecha_semana", "tienda", "producto"])
      .agg(
          pl.col("ventas").sum().alias("ventas")
      )
)

df_limpio.head()

fecha_semana,tienda,producto,ventas
datetime[ns],i16,i32,f32
2014-06-02 00:00:00,50,903285,139.830002
2014-10-13 00:00:00,48,1082907,6.0
2017-05-22 00:00:00,46,1456939,6.0
2017-06-19 00:00:00,52,634009,16.330999
2013-04-01 00:00:00,35,1087269,13.0


## 13. Definir la unidad de análisis

Cada fila representa las ventas semanales de un producto en una tienda.

Para describir el comportamiento de una tienda, necesitamos una nueva unidad de análisis:

> **Tienda + Fecha de fecha de semana**

In [12]:
observaciones = (
    df_limpio.select(["fecha_semana", "tienda"])
             .unique()
             .height
)

print("Observaciones tienda-semana:", observaciones)

Observaciones tienda-semana: 12060


## 14. Seleccionar los productos más frecuentes

Para controlar el tamaño de la matriz, seleccionaremos hasta 5.000 productos.

Los productos se ordenan según la cantidad de registros en los que aparecen.

In [13]:
MAX_PRODUCTOS = 5000

productos_seleccionados = (
    df_limpio.group_by("producto")
             .len()
             .sort("len", descending=True)
             .head(MAX_PRODUCTOS)
             .get_column("producto")
             .to_list()
)

print("Productos seleccionados:", len(productos_seleccionados))

Productos seleccionados: 4036


## 15. Filtrar los productos seleccionados

In [14]:
df_filtrado = df_limpio.filter(
    pl.col("producto").is_in(productos_seleccionados)
)

df_filtrado.shape

(23582329, 4)

## 16. Construir la matriz de características

Aplicamos un **pivot**:

- filas: fecha de fecha_fecha de semana + tienda;
- columnas: productos;
- valores: ventas.

Los productos dejan de ser registros y pasan a ser características.

In [15]:
matriz = (
    df_filtrado.pivot(
        values="ventas",
        index=["fecha_semana", "tienda"],
        on="producto",
        aggregate_function="sum"
    )
    .fill_null(0)
    .sort(["fecha_semana", "tienda"])
)

matriz.head()

fecha_semana,tienda,903285,1082907,1456939,634009,1087269,1336461,502182,829658,265237,584103,583975,1473482,1463993,751759,687549,1057490,377978,996606,1105212,269029,796395,273528,1050370,857332,514597,1309246,1080756,1966995,454589,730258,584188,313641,1104725,750855,1209718,…,2011450,2116132,2121690,2056876,2123775,2109909,2003392,2056764,2126842,2123036,2011457,2114752,2126944,2075374,2116139,2123711,2011448,2123839,2122868,2011437,2123790,2011470,1976284,2116238,2011468,2123791,2011451,2123209,2122947,2121610,2015898,2123747,2123863,2011459,2123859,2122818,2123463
datetime[ns],i16,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
2012-12-31 00:00:00,1,39.110001,10.0,0.0,25.788,0.0,0.0,0.0,0.0,37.0,0.0,0.0,0.0,0.0,0.0,2.0,9.0,10.0,28.744999,144.0,78.0,21.0,64.0,0.0,0.0,9.0,0.0,1.0,0.0,33.0,0.0,9.047,0.0,7.0,2.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2012-12-31 00:00:00,2,66.300003,0.0,0.0,83.259995,0.0,0.0,8.0,0.0,45.0,0.0,0.0,0.0,0.0,0.0,6.0,20.0,5.0,19.664,71.0,89.0,30.0,99.0,0.0,0.0,46.0,0.0,16.0,0.0,8.0,0.0,23.693001,14.0,59.0,9.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2012-12-31 00:00:00,3,302.062988,94.0,0.0,117.904999,0.0,0.0,32.0,0.0,202.0,0.0,0.0,0.0,0.0,0.0,2.0,14.0,16.0,62.750999,230.0,193.0,95.0,190.0,9.0,0.0,34.0,0.0,12.0,0.0,41.0,9.0,28.747,18.0,15.0,12.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2012-12-31 00:00:00,4,108.912994,21.0,0.0,26.875999,0.0,0.0,11.0,0.0,79.0,0.0,0.0,0.0,0.0,0.0,8.0,12.0,3.0,14.615,47.0,39.0,23.0,85.0,2.0,0.0,33.0,0.0,12.0,0.0,15.0,0.0,27.860001,18.0,0.0,1.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2012-12-31 00:00:00,5,18.094,5.0,0.0,107.593002,0.0,0.0,30.0,0.0,71.0,0.0,0.0,0.0,0.0,0.0,20.0,26.0,8.0,8.741,96.0,25.0,36.0,260.0,2.0,0.0,19.0,0.0,24.0,0.0,10.0,0.0,0.0,15.0,8.0,16.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 17. Dimensiones de la matriz

In [16]:
print("Observaciones:", matriz.height)
print("Variables totales:", matriz.width)
print("Productos como características:", matriz.width - 2)

Observaciones: 12060
Variables totales: 4038
Productos como características: 4036


## 18. Separar identificadores y características

Las columnas `fecha_semana` y `tienda` identifican cada observación.

Las columnas correspondientes a productos constituyen la matriz de características.

Hasta aquí trabajamos con **Polars** para cargar, limpiar y transformar los datos.  
A partir de este punto convertiremos solamente la matriz de características a un **DataFrame de pandas**, porque `scikit-learn` se integra de manera natural con esta estructura.

In [17]:
identificadores = matriz.select(["fecha_semana", "tienda"])

df_features = (
    matriz
    .drop(["fecha_semana", "tienda"])
    .to_pandas()
)

print("Tipo de objeto:", type(df_features))
print("Forma de la matriz de características:", df_features.shape)

df_features.head()

Tipo de objeto: <class 'pandas.core.frame.DataFrame'>
Forma de la matriz de características: (12060, 4036)


,903285,1082907,1456939,634009,1087269,1336461,502182,829658,265237,584103,...,2123209,2122947,2121610,2015898,2123747,2123863,2011459,2123859,2122818,2123463
0,39.110001,10.0,0.0,25.788000,0.0,0.0,0.0,0.0,37.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,66.300003,0.0,0.0,83.259995,0.0,0.0,8.0,0.0,45.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,302.062988,94.0,0.0,117.904999,0.0,0.0,32.0,0.0,202.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,108.912994,21.0,0.0,26.875999,0.0,0.0,11.0,0.0,79.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,18.094000,5.0,0.0,107.593002,0.0,0.0,30.0,0.0,71.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 19. Analizar la dispersión de la matriz

En una semana determinada, no todos los productos se venden en todas las tiendas.

Por esta razón, muchas celdas contienen el valor cero. Mientras mayor sea el porcentaje de ceros, más dispersa será la matriz.

In [18]:
porcentaje_ceros = (
    df_features.eq(0).to_numpy().mean() * 100
)

filas, columnas = df_features.shape

print(f"Observaciones              : {filas:,}")
print(f"Productos                  : {columnas:,}")
print(f"Porcentaje de valores cero : {porcentaje_ceros:.2f}%")
print(f"Densidad de la matriz      : {100 - porcentaje_ceros:.2f}%")

Observaciones              : 12,060
Productos                  : 4,036
Porcentaje de valores cero : 51.55%
Densidad de la matriz      : 48.45%


## 20. Escalar los datos

PCA es sensible a la escala de las variables.

Algunos productos venden pocas unidades por semana, mientras que otros pueden vender cientos o miles. Sin escalamiento, los productos con mayores magnitudes dominarían el cálculo de los componentes principales.

`StandardScaler` transforma cada producto para que tenga media 0 y desviación estándar 1.

El resultado se reconstruye como un **DataFrame de pandas**, de modo que los nombres de los productos se conserven visibles.

In [19]:
scaler = StandardScaler()

df_escalado = pd.DataFrame(
    scaler.fit_transform(df_features),
    columns=df_features.columns,
    index=df_features.index
)

print("Tipo de objeto:", type(df_escalado))
print("Forma de la matriz escalada:", df_escalado.shape)

df_escalado.head()

Tipo de objeto: <class 'pandas.core.frame.DataFrame'>
Forma de la matriz escalada: (12060, 4036)


,903285,1082907,1456939,634009,1087269,1336461,502182,829658,265237,584103,...,2123209,2122947,2121610,2015898,2123747,2123863,2011459,2123859,2122818,2123463
0,-0.654452,-0.178381,-0.36441,-0.006643,-0.997462,-0.199333,-0.931950,-0.689605,-0.141682,-0.65189,...,-0.014604,-0.020366,-0.021863,-0.009106,-0.012218,-0.023083,-0.021266,-0.016814,-0.011519,-0.013301
1,-0.555878,-0.445067,-0.36441,1.461163,-0.997462,-0.199333,-0.405000,-0.689605,0.044100,-0.65189,...,-0.014604,-0.020366,-0.021863,-0.009106,-0.012218,-0.023083,-0.021266,-0.016814,-0.011519,-0.013301
2,0.298848,2.061782,-0.36441,2.345978,-0.997462,-0.199333,1.175850,-0.689605,3.690063,-0.65189,...,-0.014604,-0.020366,-0.021863,-0.009106,-0.012218,-0.023083,-0.021266,-0.016814,-0.011519,-0.013301
3,-0.401390,0.114974,-0.36441,0.021144,-0.997462,-0.199333,-0.207394,-0.689605,0.833672,-0.65189,...,-0.014604,-0.020366,-0.021863,-0.009106,-0.012218,-0.023083,-0.021266,-0.016814,-0.011519,-0.013301
4,-0.730642,-0.311724,-0.36441,2.082615,-0.997462,-0.199333,1.044112,-0.689605,0.647890,-0.65189,...,-0.014604,-0.020366,-0.021863,-0.009106,-0.012218,-0.023083,-0.021266,-0.016814,-0.011519,-0.013301


## 21. Aplicar PCA

Aplicamos PCA directamente sobre el DataFrame escalado.

Primero calculamos todos los componentes posibles para analizar la varianza explicada. Aunque `scikit-learn` realiza internamente operaciones numéricas, durante el laboratorio mantendremos los resultados organizados como DataFrames.

In [20]:
pca = PCA()

componentes_completos = pca.fit_transform(
    df_escalado
)

varianza_acumulada = np.cumsum(
    pca.explained_variance_ratio_
)

## 22. Graficar interactivamente la varianza explicada acumulada

La línea horizontal muestra el objetivo de conservar el 95 % de la información.

In [21]:
n_componentes = np.argmax(
    varianza_acumulada >= 0.95
) + 1

df_varianza = pl.DataFrame({
    "componente": np.arange(1, len(varianza_acumulada) + 1),
    "varianza_acumulada": varianza_acumulada
})

fig = px.line(
    x=df_varianza["componente"].to_numpy(),
    y=df_varianza["varianza_acumulada"].to_numpy(),
    markers=True,
    title="Selección del número de componentes",
    labels={
        "x": "Número de componentes",
        "y": "Varianza explicada acumulada"
    }
)

fig.add_hline(
    y=0.95,
    line_dash="dash",
    annotation_text="95 % de varianza",
    annotation_position="bottom right"
)

fig.update_layout(
    hovermode="x unified"
)

fig.add_vline(
    x=n_componentes,
    line_dash="dash",
    line_color="red"
)

fig.add_annotation(
    x=n_componentes + 120,
    y=1.01,
    text=f"{n_componentes} componentes",
    showarrow=True,
    arrowhead=2,
    arrowcolor="red",
    font=dict(color="red")
)

fig.show()

## 23. Determinar cuántos componentes conservan el 95 %

El número no se elige arbitrariamente.  
Se obtiene directamente desde la curva de varianza explicada.

In [22]:
n_componentes = np.argmax(
    varianza_acumulada >= 0.95
) + 1

print("Componentes necesarios:", n_componentes)
print(
    "Varianza conservada:",
    f"{varianza_acumulada[n_componentes - 1]:.2%}"
)

Componentes necesarios: 1441
Varianza conservada: 95.00%


## 24. Construir la matriz reducida

Aplicamos nuevamente PCA utilizando solamente el número de componentes seleccionado.

El resultado se guarda en un nuevo DataFrame cuyas columnas son:

`PC1`, `PC2`, `PC3`, ..., `PCn`.

In [23]:
pca_95 = PCA(
    n_components=n_componentes
)

componentes_reducidos = pca_95.fit_transform(
    df_escalado
)

columnas_pca = [
    f"PC{i}"
    for i in range(1, n_componentes + 1)
]

df_pca = pd.DataFrame(
    componentes_reducidos,
    columns=columnas_pca,
    index=df_escalado.index
)

print("Matriz original :", df_features.shape)
print("Matriz reducida :", df_pca.shape)

df_pca.head()

Matriz original : (12060, 4036)
Matriz reducida : (12060, 1441)


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC1432,PC1433,PC1434,PC1435,PC1436,PC1437,PC1438,PC1439,PC1440,PC1441
0,-18.349596,-13.859055,-14.018353,-5.002702,3.926944,-1.485696,-4.596514,-6.407011,-0.995503,1.434670,...,0.162419,-0.082416,-0.034838,-0.055364,0.227614,0.043512,-0.102144,0.115974,0.051295,0.005450
1,-12.590069,-17.580343,-6.182944,-9.035910,3.316004,-0.657043,-5.541713,-3.080096,-2.536962,2.787453,...,-0.280164,-0.280517,-0.239836,-0.137373,0.357761,-0.105702,-0.037713,-0.266949,0.366523,-0.511929
2,14.253777,-37.511730,8.605852,-16.583349,-13.201539,-0.930083,-16.992445,-5.351492,-5.134505,0.827390,...,-0.237493,-0.408718,-0.854871,-0.570194,-0.382261,-0.661617,0.484233,0.214699,-0.509136,-0.201647
3,-12.065720,-17.950672,-5.689428,-9.334553,3.078813,-0.509023,-5.404881,-3.238324,-2.470276,2.515188,...,0.565223,-0.229289,-0.122626,-0.241715,0.270746,-0.067404,0.234404,0.685770,0.220762,0.011661
4,-13.914412,-16.096479,-4.293474,-9.626466,4.478323,3.580824,-2.446550,2.824054,-0.765581,1.409436,...,-0.037204,-0.229092,-0.354301,-0.722338,-0.407800,-0.140221,-0.306730,-0.077053,-0.509907,-0.455524


## 25. Incorporar los identificadores

Unimos nuevamente:

- `fecha_semana`;
- `tienda`;
- componentes principales.

Así obtenemos una tabla lista para exportar y utilizar en módulos posteriores.

In [24]:
df_componentes = pl.from_pandas(
    df_pca
)

resultado_pca = pl.concat(
    [identificadores, df_componentes],
    how="horizontal"
)

resultado_pca.head()

fecha_semana,tienda,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,PC21,PC22,PC23,PC24,PC25,PC26,PC27,PC28,PC29,PC30,PC31,PC32,PC33,PC34,PC35,…,PC1405,PC1406,PC1407,PC1408,PC1409,PC1410,PC1411,PC1412,PC1413,PC1414,PC1415,PC1416,PC1417,PC1418,PC1419,PC1420,PC1421,PC1422,PC1423,PC1424,PC1425,PC1426,PC1427,PC1428,PC1429,PC1430,PC1431,PC1432,PC1433,PC1434,PC1435,PC1436,PC1437,PC1438,PC1439,PC1440,PC1441
datetime[ns],i16,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
2012-12-31 00:00:00,1,-18.349596,-13.859055,-14.018353,-5.002702,3.926944,-1.485696,-4.596514,-6.407011,-0.995503,1.43467,1.840649,-0.540375,1.154332,-1.532719,-3.385542,-1.47853,-0.113035,0.489592,1.608084,2.207357,0.356034,0.663702,-1.216719,-0.616479,-0.646586,0.555498,-1.605494,-0.978255,-0.368378,0.013947,-0.443086,-0.112575,-0.042651,0.197588,0.972582,…,-0.434893,0.005512,0.194736,0.402887,0.051248,0.103092,-0.134743,0.488009,-0.00517,0.033368,0.108314,-0.433667,-0.39869,-0.047257,-0.045741,0.003839,0.380568,0.078571,-0.188167,-0.250068,0.280413,0.081482,0.008407,-0.31232,-0.246334,0.236644,-0.315707,0.162419,-0.082416,-0.034838,-0.055364,0.227614,0.043512,-0.102144,0.115974,0.051295,0.00545
2012-12-31 00:00:00,2,-12.590069,-17.580343,-6.182944,-9.03591,3.316004,-0.657043,-5.541713,-3.080096,-2.536962,2.787453,5.384452,-1.062303,-0.457871,-2.213508,-3.023008,-2.750483,-0.447015,2.081613,1.166002,3.169561,-2.152524,1.104686,-1.192018,-0.154998,-0.044071,1.224549,-0.604087,-1.424081,0.29196,-0.288428,-0.108757,0.669717,0.119342,-0.505112,1.089963,…,0.089631,0.032694,-0.390196,0.456211,-0.170884,-0.451944,0.148916,0.4059,-0.147174,-0.064577,-0.165185,-0.007413,0.072378,-0.607412,-0.032007,-0.020343,0.487975,0.161813,-0.096797,-0.164724,-0.14866,-0.554784,0.29054,-0.087482,0.360968,-0.007657,0.051217,-0.280164,-0.280517,-0.239836,-0.137373,0.357761,-0.105702,-0.037713,-0.266949,0.366523,-0.511929
2012-12-31 00:00:00,3,14.253777,-37.51173,8.605852,-16.583349,-13.201539,-0.930083,-16.992445,-5.351492,-5.134505,0.82739,13.508747,0.719128,-3.010841,-7.511698,-2.573537,-5.21047,-4.684591,0.712505,9.165178,2.360857,-5.404414,6.596986,-0.960722,0.847814,2.006785,0.5321,-0.644339,0.411153,0.497941,5.39543,-1.077348,-3.749046,3.072455,-0.015283,0.937113,…,0.238632,0.813576,0.248534,0.911431,0.516705,-0.372914,-0.240352,-0.554394,0.28141,-0.266949,-0.081348,-0.105706,0.185918,-0.213298,0.266253,-0.225331,0.64251,0.148629,0.530257,0.043355,-0.129898,0.452567,0.485271,-0.573049,-0.052474,0.72422,0.340766,-0.237493,-0.408718,-0.854871,-0.570194,-0.382261,-0.661617,0.484233,0.214699,-0.509136,-0.201647
2012-12-31 00:00:00,4,-12.06572,-17.950672,-5.689428,-9.334553,3.078813,-0.509023,-5.404881,-3.238324,-2.470276,2.515188,5.476717,-1.017123,-0.320004,-2.850763,-1.566736,-3.101889,0.013028,2.02753,1.404717,2.985376,-1.642596,0.880036,-1.607589,-0.814394,0.252778,0.904337,-1.069959,-0.891623,0.415847,0.011621,-0.029969,0.580304,0.773284,-0.091032,1.239394,…,-0.272078,0.013159,-0.178858,0.191979,-0.200579,0.419981,-0.040166,-0.153448,-0.240232,-0.244866,-0.5381,-0.316624,0.439468,-0.319524,0.182233,0.073014,0.351321,-0.148059,-0.006826,-0.039112,0.467993,-0.084325,-0.02295,0.669694,-0.027987,0.341336,0.387548,0.565223,-0.229289,-0.122626,-0.241715,0.270746,-0.067404,0.234404,0.68577,0.220762,0.011661
2012-12-31 00:00:00,5,-13.914412,-16.096479,-4.293474,-9.626466,4.478323,3.580824,-2.44655,2.824054,-0.765581,1.409436,7.622374,1.467529,-2.003405,-0.65461,2.21428,-1.8562,-4.131773,4.947376,1.620393,1.73644,-0.628578,4.537565,-2.961912,-5.368043,1.917309,0.939717,-2.588606,-2.667497,-0.398848,-3.2392,2.417156,-0.414188,0.990428,2.533376,-0.416219,…,-0.353378,0.163397,-0.075478,0.311787,-0.629049,-0.051269,

## 26. PCA con dos componentes para visualización

Dos componentes se utilizan solamente para representar los datos en un plano.

El resultado también se organiza como un DataFrame con las columnas `PC1` y `PC2`.

Esto no significa que dos componentes conserven necesariamente el 95 % de la información.

In [25]:
pca_2d = PCA(
    n_components=2
)

componentes_2d = pca_2d.fit_transform(
    df_escalado
)

df_pca_2d = pd.DataFrame(
    componentes_2d,
    columns=["PC1", "PC2"],
    index=df_escalado.index
)

print(
    "Varianza conservada por PC1 y PC2:",
    f"{pca_2d.explained_variance_ratio_.sum():.2%}"
)

df_pca_2d.head()

Varianza conservada por PC1 y PC2: 30.53%


,PC1,PC2
0,-18.349564,-13.858690
1,-12.590053,-17.580420
2,14.253826,-37.511730
3,-12.065693,-17.950666
4,-13.914388,-16.096481


## 27. Visualizar interactivamente las observaciones

Cada punto representa una combinación tienda-fecha de fecha de semana.

Los puntos cercanos muestran patrones de venta similares.

In [26]:
df_visualizacion = pd.DataFrame({
    "PC1": df_pca_2d["PC1"],
    "PC2": df_pca_2d["PC2"],
    "tienda": identificadores.get_column("tienda").to_list(),
    "fecha_semana": identificadores.get_column("fecha_semana").cast(pl.String).to_list()
})

fig = px.scatter(
    df_visualizacion,
    x="PC1",
    y="PC2",
    hover_data=["tienda", "fecha_semana"],
    title="Tiendas-semana proyectadas mediante PCA",
    labels={
        "PC1": "Componente principal 1",
        "PC2": "Componente principal 2"
    },
    opacity=0.65
)

fig.show()

Cada punto representa el comportamiento de una tienda durante una semana, descrito originalmente por más de 4.000 productos y proyectado sobre los dos primeros componentes principales. La proximidad entre puntos indica similitud en sus patrones de venta, aunque esta representación conserva solo una parte de la variabilidad total del conjunto de datos.

## 28. Visualizar interactivamente las observaciones según la tienda

In [27]:
df_visualizacion["tienda"] = (
    df_visualizacion["tienda"].astype(str)
)

fig = px.scatter(
    df_visualizacion,
    x="PC1",
    y="PC2",
    color="tienda",
    hover_data=["fecha_semana"],
    title="Observaciones diferenciadas por tienda",
    labels={
        "PC1": "Componente principal 1",
        "PC2": "Componente principal 2",
        "tienda": "Tienda"
    },
    opacity=0.7
)

fig.show()

Ver por años

In [30]:
# Crear DataFrame para la visualización
df_visualizacion = pd.DataFrame({
    "PC1": df_pca_2d["PC1"],
    "PC2": df_pca_2d["PC2"],
    "fecha_semana": identificadores.get_column("fecha_semana").to_list()
})

# Convertir a fecha y extraer el año
df_visualizacion["fecha_semana"] = pd.to_datetime(
    df_visualizacion["fecha_semana"]
)

df_visualizacion["Año"] = (
    df_visualizacion["fecha_semana"].dt.year.astype(str)
)

# Gráfico interactivo
fig = px.scatter(
    df_visualizacion,
    x="PC1",
    y="PC2",
    color="Año",
    hover_data=["fecha_semana"],
    color_discrete_sequence=px.colors.sequential.Viridis,
    title="Proyección PCA de las observaciones coloreadas por año",
    labels={
        "PC1": "Componente principal 1",
        "PC2": "Componente principal 2"
    },
    opacity=0.7
)

fig.show()

## **Interpretación de la proyección por año**

La siguiente visualización muestra las observaciones proyectadas sobre los dos primeros componentes principales y coloreadas según el año correspondiente.

Podemos observar que:

- Los datos de los distintos años presentan una amplia superposición, por lo que no existe una separación claramente definida entre ellos.
- Se aprecia una evolución gradual de los colores, lo que sugiere que el comportamiento de las ventas cambia de manera progresiva a lo largo del tiempo y no mediante cambios bruscos.
- Las observaciones correspondientes a todos los años ocupan gran parte del espacio proyectado, indicando que existe una alta variabilidad en los patrones de venta dentro de cada año.
- Esta representación utiliza únicamente los dos primeros componentes principales, por lo que resulta útil para explorar visualmente los datos, pero no conserva toda la información del conjunto original.

**Conclusión:**

La proyección mediante PCA permite visualizar la estructura general de los datos y explorar posibles tendencias temporales, aunque por sí sola no permite afirmar la existencia de grupos claramente diferenciados entre los años.

In [31]:
# Crear DataFrame para la visualización
df_visualizacion = pd.DataFrame({
    "PC1": df_pca_2d["PC1"],
    "PC2": df_pca_2d["PC2"],
    "fecha_semana": identificadores.get_column("fecha_semana").to_list()
})

# Convertir a fecha
df_visualizacion["fecha_semana"] = pd.to_datetime(
    df_visualizacion["fecha_semana"]
)

# Extraer el mes
meses = {
    1: "Enero",
    2: "Febrero",
    3: "Marzo",
    4: "Abril",
    5: "Mayo",
    6: "Junio",
    7: "Julio",
    8: "Agosto",
    9: "Septiembre",
    10: "Octubre",
    11: "Noviembre",
    12: "Diciembre"
}

df_visualizacion["Mes"] = (
    df_visualizacion["fecha_semana"]
    .dt.month
    .map(meses)
)

# Orden correcto de los meses
orden_meses = list(meses.values())

# Gráfico
fig = px.scatter(
    df_visualizacion,
    x="PC1",
    y="PC2",
    color="Mes",
    category_orders={"Mes": orden_meses},
    hover_data=["fecha_semana"],
    title="Proyección PCA de las observaciones coloreadas por mes",
    labels={
        "PC1": "Componente principal 1",
        "PC2": "Componente principal 2"
    },
    opacity=0.7
)

fig.update_layout(
    legend_title="Mes"
)

fig.show()

## **Interpretación de la proyección por mes**

La figura muestra las observaciones proyectadas sobre los dos primeros componentes principales y coloreadas según el mes del año.

**Podemos observar que:**

- Los doce meses aparecen ampliamente superpuestos, sin formar regiones claramente diferenciadas.
- No se aprecia una separación evidente entre meses específicos, lo que indica que la estacionalidad no es el principal factor que explica la variabilidad capturada por los dos primeros componentes principales.
- Esto no significa que la estacionalidad no exista, sino que su efecto puede ser menor que otros factores, como las diferencias entre tiendas, promociones, categorías de productos o comportamientos particulares de compra.
- Además, la visualización utiliza únicamente dos componentes principales, mientras que para conservar aproximadamente el 95 % de la información del conjunto de datos fueron necesarios 1.441 componentes.

**Conclusión:**

La proyección mediante PCA es una herramienta útil para explorar visualmente los datos, pero la ausencia de una separación clara por meses indica que los dos primeros componentes no capturan de forma predominante la estacionalidad del negocio.

## 29. Exportar la matriz reducida

Guardamos la representación que conserva al menos el 95 % de la varianza.

In [33]:
# CSV (interoperabilidad)
resultado_pca.write_csv("favorita_pca_95.csv")

# Parquet (Big Data)
resultado_pca.write_parquet("favorita_pca_95.parquet")

print("Archivos exportados correctamente.")

Archivos exportados correctamente.


In [36]:
import os

csv_size = os.path.getsize("favorita_pca_95.csv") / (1024**2)
parquet_size = os.path.getsize("favorita_pca_95.parquet") / (1024**2)

reduccion = 100 * (1 - parquet_size / csv_size)
print(f"CSV     : {csv_size:.2f} MB")
print(f"Parquet : {parquet_size:.2f} MB")
print(f"Reducción de tamaño: {reduccion:.1f}%")

CSV     : 184.41 MB
Parquet : 62.04 MB
Reducción de tamaño: 66.4%


## 30. Descargar el resultado

In [37]:
from google.colab import files
files.download("favorita_pca_95.csv")
files.download("favorita_pca_95.parquet")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Conclusiones

En este laboratorio construimos una representación adecuada para aplicar técnicas de Machine Learning a un conjunto de datos de retail.

El proceso desarrollado fue el siguiente:

- Se cargó un conjunto de datos con más de **23 millones de registros transaccionales**.
- Se verificó la calidad de los datos, revisando valores nulos y registros duplicados.
- Se definió la **unidad de análisis** como una **tienda en una semana determinada**.
- Mediante una operación **Pivot** se construyó una **matriz de características**, donde:
  - cada fila representa una tienda durante una semana;
  - cada columna representa un producto;
  - cada valor corresponde a las ventas semanales de dicho producto.
- Se observó que la matriz presenta una alta dimensionalidad, con más de **4.000 variables**, y una cantidad importante de valores cero, característica habitual en problemas de retail.
- Se aplicó **StandardScaler** para estandarizar todas las variables antes de utilizar PCA.
- Se utilizó **Análisis de Componentes Principales (PCA)** para reducir la dimensionalidad del problema.

Como resultado, fue posible reducir el número de variables desde aproximadamente **4.036 productos** hasta **1.441 componentes principales**, conservando cerca del **95 % de la información** contenida en el conjunto de datos.

La nueva representación fue exportada tanto en formato **CSV** como **Parquet**.

## ¿Qué ventajas aporta PCA?

La reducción de dimensionalidad ofrece diversos beneficios para los modelos de Machine Learning:

- Reduce el número de variables de entrada.
- Disminuye el consumo de memoria.
- Reduce el tamaño de los archivos utilizados por los modelos.
- Puede disminuir los tiempos de entrenamiento.
- Reduce la redundancia y la multicolinealidad entre variables.
- Facilita la visualización de conjuntos de datos de muy alta dimensionalidad.

Sin embargo, es importante destacar que **PCA no garantiza una mejora en la precisión de los modelos predictivos**. Su principal objetivo es construir una representación más compacta de los datos. El impacto sobre el desempeño dependerá del algoritmo utilizado y deberá evaluarse experimentalmente.

En los próximos módulos del diplomado utilizaremos la matriz reducida obtenida mediante PCA para entrenar modelos de **Aprendizaje Supervisado** y **Aprendizaje No Supervisado**, comparando su desempeño con el obtenido utilizando las variables originales.